# Chapter 17 Lab — Decision Trees

Can a classifier learn a sequence of simple questions? Read [`blog.md`](<blog.md>) before running the experiments.

## Prediction

Predict Gini for a 50/50 binary node, which split should be preferred, and what happens as tree depth increases.

In [ ]:
import numpy as np

def gini(proportions):
    proportions = np.asarray(proportions, dtype=float)
    return 1.0 - np.sum(proportions ** 2)

assert np.isclose(gini([0.5, 0.5]), 0.5)
assert np.isclose(gini([1.0, 0.0]), 0.0)
print(gini([0.5,0.5]), gini([1,0]))

## Mathematics

Gini impurity is $G=1-\sum_k p_k^2$. A split is scored by weighted child impurity; gain is parent impurity minus child impurity.

In [ ]:
# Parent: 4 apartment, 4 villa, 2 farmhouse
parent = gini([0.4, 0.4, 0.2])
right = gini([0.0, 4/6, 2/6])
child = (4/10)*0.0 + (6/10)*right
gain = parent - child
assert np.isclose(parent, 0.64)
assert np.isclose(right, 4/9)
assert np.isclose(child, 0.26666666666666666)
assert np.isclose(gain, 0.37333333333333335)

In [ ]:
def split_gain(y, left_mask):
    y = np.asarray(y)
    left = y[left_mask]
    right = y[~left_mask]
    classes = np.unique(y)
    def impurity(labels):
        if len(labels) == 0:
            return 0.0
        probs = np.array([(labels == c).mean() for c in classes])
        return gini(probs)
    n = len(y)
    weighted = len(left)/n * impurity(left) + len(right)/n * impurity(right)
    return impurity(y) - weighted

y = np.array(['Apartment']*4 + ['Villa']*4 + ['Farmhouse']*2)
mask = np.array([True]*4 + [False]*6)
assert np.isclose(split_gain(y, mask), gain)

In [ ]:
import matplotlib.pyplot as plt

X = np.array([[2,8],[2,10],[3,12],[4,16],[5,18],[3,20]], dtype=float)
labels = np.array(['Apartment','Apartment','Apartment','Villa','Villa','Farmhouse'])
for label in np.unique(labels):
    pts = X[labels == label]
    plt.scatter(pts[:,0], pts[:,1], label=label)
plt.axhline(14, linestyle='--', label='candidate split')
plt.xlabel('rooms')
plt.ylabel('area (hundreds sq ft)')
plt.legend()
plt.show()

In [ ]:
# Test candidate area thresholds and choose the best gain
thresholds = [9, 11, 14, 17, 19]
best = None
for t in thresholds:
    gain_t = split_gain(labels, X[:,1] < t)
    if best is None or gain_t > best[1]:
        best = (t, gain_t)
print('best threshold, gain:', best)

In [ ]:
# Change exactly one variable: max depth of a real tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
X_big = rng.normal(size=(300, 2))
y_big = ((X_big[:,0] * X_big[:,1]) > 0).astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X_big, y_big, test_size=0.3, random_state=7)

for depth in [1, 2, 4, 8, None]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=7)
    model.fit(X_tr, y_tr)
    print(depth, model.score(X_tr, y_tr), model.score(X_te, y_te))

## Observe

The best split is the one with the largest impurity reduction. Increasing depth usually raises training accuracy, but after some point validation accuracy may stop improving or fall.

## Explain

A tree recursively chooses locally useful questions. Gini reduction measures whether the children became purer. Extra depth keeps adding smaller regions, which can eventually memorize noise.

In [ ]:
# Level 4–5 challenge
# YOUR CODE HERE
# Implement a one-level decision tree yourself: search all feature/threshold
# candidates, compute Gini gain, and return the best split.

## Reflection

- [ ] I can calculate Gini impurity by hand.
- [ ] I can explain why child impurity is weighted by node size.
- [ ] I can search candidate thresholds for a split.
- [ ] I understand how depth controls tree complexity.

Next: combine many trees and ask whether their errors can cancel.